In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ase.build import bulk
from ase.eos import EquationOfState
from ase.units import kJ

from mace.calculators import MACECalculator

# https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html
#  MODEL is MACE-MP-0 medium
MODEL = './2023-12-03-mace-128-L1_epoch-199.model'
calculator = MACECalculator(model_paths=MODEL, 
                            device='cpu',   # 'cuda' for GPU
                            default_dtype='float64')


# Initial crystal structure
atoms = bulk('Cu', 'fcc', a=3.615)
atoms.calc = calculator 

# Save the original cell
cell0 = atoms.cell.copy()

volumes = []
energies = []

# Uniformly scale the lattice vectors
# scale=1 corresponds to the initial lattice parameter
for scale in np.linspace(0.94, 1.06,  nine := 9):

    atoms.set_cell(cell0 * scale, scale_atoms=True)

    V = atoms.get_volume()
    E = atoms.get_potential_energy()

    volumes.append(V)
    energies.append(E)

    print(f"scale = {scale:.4f}, V = {V:.4f} Å^3, E = {E:.6f} eV")

# Fit E(V)
eos = EquationOfState(volumes, energies, eos='birchmurnaghan')

V0, E0, B0 = eos.fit()

# ASE returns bulk modulus in eV / Å^3
B0_GPa = B0 / kJ * 1.0e24

print()
print(f"Equilibrium volume = {V0:.6f} Å^3")
print(f"Minimum energy     = {E0:.6f} eV")
print(f"Bulk modulus       = {B0_GPa:.3f} GPa")
print(f" Experimental Bulk modulus = 140 GPa (https://en.wikipedia.org/wiki/Copper)")

# Plot EOS
# eos.plot(filename='Cu_EOS.png')
plt.show()

/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


scale = 0.9400, V = 9.8095 Å^3, E = -3.838404 eV
scale = 0.9550, V = 10.2867 Å^3, E = -3.950492 eV
scale = 0.9700, V = 10.7790 Å^3, E = -4.025318 eV
scale = 0.9850, V = 11.2869 Å^3, E = -4.068229 eV
scale = 1.0000, V = 11.8104 Å^3, E = -4.084400 eV
scale = 1.0150, V = 12.3499 Å^3, E = -4.078184 eV
scale = 1.0300, V = 12.9056 Å^3, E = -4.053351 eV
scale = 1.0450, V = 13.4776 Å^3, E = -4.013312 eV
scale = 1.0600, V = 14.0664 Å^3, E = -3.961068 eV

Equilibrium volume = 11.916377 Å^3
Minimum energy     = -4.084669 eV
Bulk modulus       = 142.957 GPa
